Your proposed summary table is strong for ICSME, and it maps well to what reviewers look for: performance, stability, and interpretability.

**Recommended Results Structure (ICSME-style)**

1. **RQ1: Which hyperparameters produce the best and most stable structures?**
2. **RQ2: How stable are learned dependencies across restarts/configs?**
3. **RQ3: What security-relevant relationships does the final BN reveal?**

If you only have space for 1-2 RQs, keep RQ1 and RQ3.

---

**Core Tables/Figures to Include**

1. **Table: Hyperparameter Tuning Summary**  
   Your idea is good. Use rows as `(mi_threshold, max_indegree)` and report selected `tabu_length`.
2. **Table: Final Model Characteristics (best config per MI)**  
   Include `n_edges`, `n_edges_on_target`, BIC, parent set of `is_vul`, top incoming edge strengths/frequencies.
3. **Figure: BIC Heatmap**  
   Axes: `tabu_length` and `max_indegree`, one panel per `mi_threshold`.
4. **Figure: Stability Plot**  
   `#restarts` vs best-so-far score and/or edge inclusion frequencies.
5. **Figure: Ego graph around `is_vul`**  
   Cleaner than full DAG for paper readability.

---

**Your Proposed Table: Refined Version**

Use this schema:

- **Rows**: `(mi_threshold, max_indegree)` combinations  
- **Columns**:
  - `selected_tabu_length`
  - `best_mean_BIC` (across restarts for selected tabu)
  - `BIC_std`
  - `n_unique_DAGs`
  - `mean_pairwise_SHD`
  - `mean_edge_count`
  - `n_restarts` (important context; usually 50)

This is publishable and concise.

---

**Small but Important Improvements**

1. Define **selection rule** clearly:  
   `selected_tabu_length = argmax mean_BIC` per `(mi_threshold, max_indegree)`.
2. Add a **tie-breaker rule**:  
   choose smaller `tabu_length`, then lower `mean_pairwise_SHD`.
3. Specify SHD variant:  
   directed SHD on DAGs (or CPDAG SHD if you use equivalence classes).
4. Report uncertainty:  
   mean ± std is fine; median + IQR can be added in appendix if skewed.

---

**Suggested Results Section Headings**

1. `5.1 Hyperparameter Sensitivity and Selection`
2. `5.2 Structural Stability Across Restarts`
3. `5.3 Interpreting Vulnerability-Related Dependencies`
4. `5.4 Robustness Checks` (optional but appreciated by reviewers)

If you want, I can draft the exact LaTeX table skeleton and a short “Results narrative template” tailored to your current `mi50/mi100` outputs.

In [15]:
from pathlib import Path
import json
import pandas as pd

from diff_analysis.utils.config_utils import find_project_root

In [16]:
project_root = find_project_root()
data_dir = project_root / "data/results/paper"
mi50_dir = data_dir / "mi50"
mi100_dir = data_dir / "mi100"

for experiment_dir in [mi50_dir, mi100_dir]:
    if not experiment_dir.exists():
        print(f"Experiment directory {experiment_dir} does not exist.")

# MI 50 experiments

In [8]:
tuning_metadata = mi50_dir / "experiment.json"


with open(tuning_metadata) as f:
    tuning_metadata = json.load(f)

tuning_metadata

{'script': 'diff_analysis.scripts.megavul.bn1.tune_bn1',
 'grid': {'mi_threshold': [50, 100, 200],
  'tabu_length': [10, 50, 100],
  'max_indegree': [None, 3, 5]},
 'hcs_params': {'delta': 0.05, 'c': 0.1, 'max_restarts': 50},
 'fixed_params': {'scoring': 'bic-d',
  'max_iter': 1000000,
  'feature_threshold': 0.0001,
  'sample_threshold': 0.0,
  'target_col': 'is_vul'},
 'n_configs': 27,
 'max_restarts_per_config': 50,
 'max_total_restarts': 1350}

Load results summary

In [14]:
results_summary = mi50_dir / 'summary.csv'
if results_summary.exists():
    results_summary = pd.read_csv(results_summary)
else:
    raise FileNotFoundError(f"Results summary {results_summary} does not exist.")

results_summary

,config_index,mi_threshold,tabu_length,max_indegree,started_at,ended_at,n_restarts_used,n_distinct_optima,n_edges,bic_score,n_edges_on_target,elapsed_s,error
0,1,50,10,NaN,2026-03-03T22:50:55,2026-03-03T22:56:38,50,45,69,-24095.95,31,342.8,NaN
1,2,50,10,3.0,2026-03-03T22:56:38,2026-03-03T23:02:09,50,45,69,-24095.95,31,330.6,NaN
2,3,50,10,5.0,2026-03-03T23:02:09,2026-03-03T23:07:53,50,45,69,-24095.95,31,344.1,NaN
3,4,50,50,NaN,2026-03-03T23:07:53,2026-03-03T23:13:34,50,45,69,-24095.95,31,341.2,NaN
4,5,50,50,3.0,2026-03-03T23:13:34,2026-03-03T23:18:59,50,45,69,-24095.95,31,324.7,NaN
5,6,50,50,5.0,2026-03-03T23:18:59,2026-03-03T23:24:34,50,45,69,-24095.95,31,335.6,NaN
6,7,50,100,NaN,2026-03-03T23:24:34,2026-03-03T23:30:16,50,45,70,-24096.53,31,341.4,NaN
7,8,50,100,3.0,2026-03-04T09:06:20,2026-03-04T09:12:05,50,48,70,-24096.53,31,344.2,NaN


In [21]:
# Get the best restart for each configuration in an experiment and compute the Structural Hamming Distance (SHD) between the resulting graphs.

import itertools
import math
import numpy as np

def load_best_restart(config_dir: Path):
    restarts_file = config_dir / "hcs_restarts.jsonl"
    if not restarts_file.exists():
        return None

    rows = []
    with open(restarts_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    if not rows:
        return None

    best = max(rows, key=lambda r: r.get("score", -math.inf))
    edges = {tuple(edge) for edge in best.get("edges", [])}
    return {
        "config": config_dir.name,
        "restart": best.get("restart"),
        "score": best.get("score"),
        "edges": edges,
    }

def directed_shd(edges_a: set[tuple[str, str]], edges_b: set[tuple[str, str]], nodes: set[str]) -> int:
    dist = 0
    for u, v in itertools.combinations(sorted(nodes), 2):
        # 0 = no edge, 1 = u->v, -1 = v->u
        a_state = 1 if (u, v) in edges_a else (-1 if (v, u) in edges_a else 0)
        b_state = 1 if (u, v) in edges_b else (-1 if (v, u) in edges_b else 0)
        dist += int(a_state != b_state)
    return dist

def shd_stats_for_experiment(experiment_dir: Path):
    configs_dir = experiment_dir / "configs"
    best_graphs = []
    for config_dir in sorted(configs_dir.glob("*")):
        if config_dir.is_dir():
            best = load_best_restart(config_dir)
            if best is not None:
                best_graphs.append(best)

    if len(best_graphs) < 2:
        raise ValueError(
            f"Need at least 2 configs with restart data in {experiment_dir.name} to compute SHD."
        )

    all_nodes = set()
    for g in best_graphs:
        for u, v in g["edges"]:
            all_nodes.add(u)
            all_nodes.add(v)

    pairwise = []
    for g1, g2 in itertools.combinations(best_graphs, 2):
        shd = directed_shd(g1["edges"], g2["edges"], all_nodes)
        pairwise.append({
            "config_a": g1["config"],
            "config_b": g2["config"],
            "shd": shd,
        })

    pairwise_df = pd.DataFrame(pairwise).sort_values("shd", ascending=False).reset_index(drop=True)
    return {
        "experiment": experiment_dir.name,
        "n_configs": len(best_graphs),
        "n_pairs": len(pairwise_df),
        "mean_shd": float(np.mean(pairwise_df["shd"])),
        "var_shd": float(np.var(pairwise_df["shd"], ddof=0)),
        "pairwise_df": pairwise_df,
    }

stats_rows = []
pairwise_tables = {}
for experiment_dir in [mi50_dir, mi100_dir]:
    stats = shd_stats_for_experiment(experiment_dir)
    pairwise_tables[experiment_dir.name] = stats.pop("pairwise_df")
    stats_rows.append(stats)

shd_summary = pd.DataFrame(stats_rows)
shd_summary

,experiment,n_configs,n_pairs,mean_shd,var_shd
0,mi50,8,28,2.035714,4.963010
1,mi100,9,36,1.166667,2.138889


In [25]:
# Per-config SHD stats across restarts: compare each non-best restart DAG to the best DAG in the same config.

import numpy as np
import math

def _read_restarts_as_edge_sets(config_dir: Path):
    restarts_file = config_dir / "hcs_restarts.jsonl"
    if not restarts_file.exists():
        return []

    rows = []
    with open(restarts_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["edges"] = {tuple(edge) for edge in row.get("edges", [])}
            rows.append(row)
    return rows

def shd_to_best_per_config(experiment_dir: Path) -> pd.DataFrame:
    records = []
    configs_dir = experiment_dir / "configs"

    for config_dir in sorted(configs_dir.glob("*")):
        if not config_dir.is_dir():
            continue

        restarts = _read_restarts_as_edge_sets(config_dir)
        if not restarts:
            continue

        # In this pipeline, score is the BIC-style objective and is maximized.
        best_idx = max(range(len(restarts)), key=lambda i: restarts[i].get("score", -math.inf))
        best = restarts[best_idx]
        best_edges = best["edges"]

        all_nodes = set()
        for r in restarts:
            for u, v in r["edges"]:
                all_nodes.add(u)
                all_nodes.add(v)

        # Exclude the selected best restart itself from SHD-to-best aggregation.
        other_restarts = [r for i, r in enumerate(restarts) if i != best_idx]
        shd_vals = [
            directed_shd(r["edges"], best_edges, all_nodes)
            for r in other_restarts
        ]

        if shd_vals:
            mean_shd = float(np.mean(shd_vals))
            var_shd = float(np.var(shd_vals, ddof=0))
            min_shd = int(np.min(shd_vals))
            max_shd = int(np.max(shd_vals))
        else:
            mean_shd = float("nan")
            var_shd = float("nan")
            min_shd = None
            max_shd = None

        records.append({
            "experiment": experiment_dir.name,
            "config": config_dir.name,
            "n_restarts": len(restarts),
            "best_restart": best.get("restart"),
            "best_score": best.get("score"),
            "mean_shd_to_best": mean_shd,
            "var_shd_to_best": var_shd,
            "min_shd_to_best": min_shd,
            "max_shd_to_best": max_shd,
        })

    return pd.DataFrame(records).sort_values(["experiment", "config"]).reset_index(drop=True)

config_shd_summary = pd.concat(
    [shd_to_best_per_config(mi50_dir), shd_to_best_per_config(mi100_dir)],
    ignore_index=True,
 )
config_shd_summary

,experiment,config,n_restarts,best_restart,best_score,mean_shd_to_best,var_shd_to_best,min_shd_to_best,max_shd_to_best
0,mi50,001_mi50_tabu10_indegNone,50,11,-24095.947053,14.632653,15.579342,4,20
1,mi50,002_mi50_tabu10_indeg3,50,11,-24095.947053,14.693878,15.192003,4,20
2,mi50,003_mi50_tabu10_indeg5,50,11,-24095.947053,14.632653,15.579342,4,20
3,mi50,004_mi50_tabu50_indegNone,50,11,-24095.947053,14.632653,15.579342,4,20
4,mi50,005_mi50_tabu50_indeg3,50,11,-24095.947053,14.693878,15.192003,4,20
5,mi50,006_mi50_tabu50_indeg5,50,11,-24095.947053,14.632653,15.579342,4,20
6,mi50,007_mi50_tabu100_indegNone,50,30,-24096.534634,13.938776,14.424823,4,22
7,mi50,008_mi50_tabu100_indeg3,71,30,-24096.534634,12.571429,17.216327,3,20
8,mi100,001_mi100_tabu10_indegNone,50,47,-28609.169673,31.387755,39.257809,22,48
9,mi100,002_mi100_tabu10_indeg3,50,47,-28609.169673,31.367347,39.211995,22,48


In [26]:
# Save both SHD result tables to disk.

output_dir = data_dir
output_dir.mkdir(parents=True, exist_ok=True)

experiment_shd_path = output_dir / "shd_summary_across_configs.csv"
config_shd_path = output_dir / "shd_summary_within_config.csv"

shd_summary.to_csv(experiment_shd_path, index=False)
config_shd_summary.to_csv(config_shd_path, index=False)

print(f"Saved: {experiment_shd_path} ({len(shd_summary)} rows)")
print(f"Saved: {config_shd_path} ({len(config_shd_summary)} rows)")

Saved: /home/zayan/Documents/code/mine/research-lm_error_analysis/data/results/paper/shd_summary_across_configs.csv (2 rows)
Saved: /home/zayan/Documents/code/mine/research-lm_error_analysis/data/results/paper/shd_summary_within_config.csv (17 rows)


In [29]:
# Condition-level pandas table (one row per MI threshold).
# This table can be exported directly to CSV/JSONL.

import numpy as np


def _load_condition_summary(condition: str) -> pd.DataFrame:
    summary_path = data_dir / condition / "summary.csv"
    if not summary_path.exists():
        raise FileNotFoundError(f"Missing summary file: {summary_path}")
    return pd.read_csv(summary_path)


def _pick_scalar(series: pd.Series):
    vals = series.dropna().tolist()
    return vals[0] if vals else np.nan


rows = []
for condition in ["mi50", "mi100"]:
    df = _load_condition_summary(condition)

    # Pairwise SHD over best DAG per config, computed earlier in the notebook.
    if "pairwise_tables" in globals() and condition in pairwise_tables:
        shd_vals = pairwise_tables[condition]["shd"].to_numpy(dtype=float)
    else:
        exp_dir = data_dir / condition
        shd_vals = shd_stats_for_experiment(exp_dir)["pairwise_df"]["shd"].to_numpy(dtype=float)

    restarts_col = "n_restarts_used" if "n_restarts_used" in df.columns else "n_restart"

    rows.append(
        {
            "condition": condition,
            "n_restarts": _pick_scalar(df[restarts_col]),
            "n_unique_dags": _pick_scalar(df["n_distinct_optima"]),
            "bic_mean": float(df["bic_score"].mean()),
            "bic_std": float(df["bic_score"].std(ddof=0)),
            "pairwise_shd_mean": float(np.mean(shd_vals)),
            "pairwise_shd_std": float(np.std(shd_vals, ddof=0)),
            "edge_count_mean": float(df["n_edges"].mean()),
            "edge_count_std": float(df["n_edges"].std(ddof=0)),
        }
    )

condition_summary = pd.DataFrame(rows)
condition_summary[["n_restarts", "n_unique_dags"]] = condition_summary[["n_restarts", "n_unique_dags"]].astype("Int64")

# Optional exports:
# out_csv = data_dir / "condition_summary.csv"
# out_jsonl = data_dir / "condition_summary.jsonl"
# condition_summary.to_csv(out_csv, index=False)
# condition_summary.to_json(out_jsonl, orient="records", lines=True)

condition_summary

,condition,n_restarts,n_unique_dags,bic_mean,bic_std,pairwise_shd_mean,pairwise_shd_std,edge_count_mean,edge_count_std
0,mi50,50,45,-24096.095000,0.251147,2.035714,2.227781,69.250000,0.433013
1,mi100,50,50,-28609.383333,0.399110,1.166667,1.462494,118.777778,0.415740
